# Extracción de la tabla maestra de repuestos
+
**Proyecto:** gestión de mantenimiento de equipos portuarios  

**Objetivo:** construir un catálogo confiable de repuestos a partir del archivo de información sin limpiar.  

**Salida:** `Informe_Maestro_Repuestos_Mantenimiento_Equipos_Portuarios.xlsx`.
+
El proceso detecta automáticamente la carpeta de datos, la hoja y la fila de encabezados. También acepta archivos cuya columna principal todavía se llame `PRODUCTO` para mantener compatibilidad con las fuentes históricas.


## 1. Librerías y configuración


In [ ]:
from pathlib import Path
import re
import unicodedata

import pandas as pd
from IPython.display import display


def localizar_carpeta_datos():
    """Localiza ETL/Data sin depender de la carpeta desde la que se abre Jupyter."""
    actual = Path.cwd().resolve()
    candidatos = []
    for base in (actual, *actual.parents):
        candidatos.extend((base / "Data", base / "ETL" / "Data"))

    for carpeta in candidatos:
        if (carpeta / "Informacion sin limpiar.xlsx").is_file():
            return carpeta

    rutas = "\n- ".join(str(ruta) for ruta in candidatos)
    raise FileNotFoundError(
        "No se encontró 'Informacion sin limpiar.xlsx'. Rutas revisadas:\n- " + rutas
    )


DATA_DIR = localizar_carpeta_datos()
ARCHIVO_ENTRADA = DATA_DIR / "Informacion sin limpiar.xlsx"
ARCHIVO_SALIDA = (
    DATA_DIR / "Informe_Maestro_Repuestos_Mantenimiento_Equipos_Portuarios.xlsx"
)

print(f"Entrada: {ARCHIVO_ENTRADA}")
print(f"Salida:  {ARCHIVO_SALIDA}")


## 2. Detección y carga de la tabla origen
+
Se busca una hoja y una fila de encabezado que contengan `REPUESTO` o `PRODUCTO`, junto con las cuatro columnas de clasificación.


In [ ]:
def normalizar_encabezado(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    return re.sub(r"\s+", " ", texto).strip().upper()


COLUMNAS_CLASIFICACION = [
    "CLASIFICACION I",
    "CLASIFICACION II",
    "CLASIFICACION III",
    "CLASIFICACION IV",
]


def detectar_estructura_excel(ruta, max_filas_encabezado=15):
    """Retorna (hoja, fila_encabezado, columna_principal)."""
    libro = pd.ExcelFile(ruta)

    for hoja in libro.sheet_names:
        muestra = pd.read_excel(ruta, sheet_name=hoja, header=None, nrows=max_filas_encabezado)
        for fila, valores in muestra.iterrows():
            encabezados = {normalizar_encabezado(valor) for valor in valores if pd.notna(valor)}
            principal = next(
                (nombre for nombre in ("REPUESTO", "PRODUCTO") if nombre in encabezados),
                None,
            )
            if principal and set(COLUMNAS_CLASIFICACION).issubset(encabezados):
                return hoja, fila, principal

    raise ValueError(
        "No se encontró una tabla válida. Debe incluir REPUESTO (o PRODUCTO) "
        "y CLASIFICACION I, II, III y IV dentro de las primeras "
        f"{max_filas_encabezado} filas de alguna hoja."
    )


HOJA_ORIGEN, FILA_ENCABEZADO, COLUMNA_PRINCIPAL = detectar_estructura_excel(
    ARCHIVO_ENTRADA
)

df_origen = pd.read_excel(
    ARCHIVO_ENTRADA,
    sheet_name=HOJA_ORIGEN,
    header=FILA_ENCABEZADO,
)
df_origen.columns = [normalizar_encabezado(columna) for columna in df_origen.columns]

print(f"Hoja detectada: {HOJA_ORIGEN}")
print(f"Fila de encabezado detectada (base 0): {FILA_ENCABEZADO}")
print(f"Columna principal detectada: {COLUMNA_PRINCIPAL}")
print(f"Registros cargados: {len(df_origen):,}")
display(df_origen.head())


## 3. Limpieza y estandarización de repuestos


In [ ]:
columnas_requeridas = [COLUMNA_PRINCIPAL, *COLUMNAS_CLASIFICACION]
columnas_faltantes = [
    columna for columna in columnas_requeridas if columna not in df_origen.columns
]
if columnas_faltantes:
    raise ValueError(f"Faltan columnas requeridas: {columnas_faltantes}")

df_repuestos = df_origen[columnas_requeridas].copy()

for columna in columnas_requeridas:
    df_repuestos[columna] = (
        df_repuestos[columna]
        .astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

detalle = df_repuestos[COLUMNA_PRINCIPAL]
df_repuestos["CODIGO_REPUESTO"] = (
    detalle.str.extract(r"^\s*\[([^\]]+)\]", expand=False)
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)
df_repuestos["NOMBRE_REPUESTO"] = (
    detalle.str.replace(r"^\s*\[[^\]]+\]\s*", "", regex=True)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Los códigos puramente numéricos conservan al menos cuatro dígitos.
es_numerico = df_repuestos["CODIGO_REPUESTO"].str.fullmatch(r"\d+", na=False)
df_repuestos.loc[es_numerico, "CODIGO_REPUESTO"] = (
    df_repuestos.loc[es_numerico, "CODIGO_REPUESTO"].str.zfill(4)
)

for columna in COLUMNAS_CLASIFICACION:
    df_repuestos[columna] = (
        df_repuestos[columna]
        .replace(r"^\s*$", pd.NA, regex=True)
        .fillna("NO APLICA")
    )

df_repuestos = df_repuestos[
    df_repuestos["CODIGO_REPUESTO"].notna()
    & df_repuestos["NOMBRE_REPUESTO"].notna()
    & df_repuestos["CODIGO_REPUESTO"].ne("")
    & df_repuestos["NOMBRE_REPUESTO"].ne("")
].copy()

columnas_salida = [
    "CODIGO_REPUESTO",
    "NOMBRE_REPUESTO",
    *COLUMNAS_CLASIFICACION,
]
df_maestro_repuestos = (
    df_repuestos[columnas_salida]
    .drop_duplicates(subset=["CODIGO_REPUESTO"], keep="first")
    .sort_values(["CODIGO_REPUESTO", "NOMBRE_REPUESTO"])
    .reset_index(drop=True)
)

if df_maestro_repuestos.empty:
    raise ValueError("La limpieza no produjo registros válidos de repuestos.")

print(f"Repuestos válidos y únicos: {len(df_maestro_repuestos):,}")
display(df_maestro_repuestos.head())


## 4. Controles de calidad y exportación


In [ ]:
duplicados = df_maestro_repuestos["CODIGO_REPUESTO"].duplicated().sum()
nulos_clave = df_maestro_repuestos[
    ["CODIGO_REPUESTO", "NOMBRE_REPUESTO"]
].isna().sum().sum()

if duplicados:
    raise ValueError(f"Se detectaron {duplicados} códigos de repuesto duplicados.")
if nulos_clave:
    raise ValueError(f"Se detectaron {nulos_clave} valores nulos en campos clave.")

resumen = pd.DataFrame(
    {
        "INDICADOR_MANTENIMIENTO": [
            "Repuestos catalogados",
            "Códigos únicos",
            "Categorías principales",
        ],
        "VALOR": [
            len(df_maestro_repuestos),
            df_maestro_repuestos["CODIGO_REPUESTO"].nunique(),
            df_maestro_repuestos["CLASIFICACION I"].nunique(),
        ],
    }
)

with pd.ExcelWriter(ARCHIVO_SALIDA, engine="openpyxl") as writer:
    df_maestro_repuestos.to_excel(
        writer,
        sheet_name="Repuestos_Mantenimiento",
        index=False,
    )
    resumen.to_excel(writer, sheet_name="Resumen_Mantenimiento", index=False)

print(f"Informe generado correctamente: {ARCHIVO_SALIDA}")
display(resumen)


## Resultado
+
El notebook genera `Informe_Maestro_Repuestos_Mantenimiento_Equipos_Portuarios.xlsx` con dos hojas:
+
- `Repuestos_Mantenimiento`: catálogo depurado para la gestión de mantenimiento.

- `Resumen_Mantenimiento`: indicadores básicos de control del catálogo.
